# 04 — Filtros v4 (con detector de metáfora institucional)

Iteración sobre `03_filtros_ajustados.ipynb`. Añade el **ajuste 4** para combatir el ruido detectado en la muestra del v3:

- "Social Security is a Ponzi scheme"
- "Stock market is a Ponzi"
- "AI is a ponzi scheme"
- "Marriage is a Ponzi"
- "Housing market is a ponzi"
- "Capitalism is a scam"
- ...

Estos tweets usan `ponzi`/`scam`/`fraud` como insulto retórico hacia sistemas institucionales, no como descripción de fraude real.

## Lógica del filtro

Un tweet es "metáfora institucional" si menciona uno de los sistemas listados **a ≤6 palabras** de `ponzi`/`scheme`/`scam`/`fraud`. La proximidad léxica es clave: si las palabras están muy separadas, probablemente el tweet sí esté describiendo un fraude real que tangencialmente menciona a uno de esos sistemas.

Este notebook reutiliza el CSV `scam_us_raw_dedup.csv` (no toca API).


In [ ]:
import pandas as pd
import re

pd.set_option('display.max_colwidth', None)

df_dedup = pd.read_csv("../data/raw/scam_us_raw_dedup.csv")
print(f"Tweets cargados: {len(df_dedup)}")


## Filtros v3 (mantenidos del notebook anterior)

In [ ]:
# Términos fuertes de fraude (sin ellos hay que tener contexto monetario)
STRONG_FRAUD_TERMS = re.compile(
    r"\b(scam|scammed|scammer|scammers|"
    r"fraud|fraudster|defrauded|"
    r"phishing|smishing|vishing|"
    r"ponzi|rug\s*pull|pig\s*butchering|"
    r"identity\s*theft|wire\s*fraud|"
    r"impersonat|spoof|"
    r"fake\s*(?:text|email|delivery|invoice|website|caller))\b",
    re.IGNORECASE,
)

MONEY_TERMS = re.compile(
    r"(\b(money|cash|dollar|dollars|usd|paid|payment|sent|wired|wire|"
    r"transfer|refund|bank|account|credit\s*card|debit|wallet|invoice|"
    r"charged|stolen|lost|victim|defrauded|deposit|withdraw|"
    r"check|venmo|zelle|paypal|crypto|bitcoin|coinbase|gift\s*card)\b|"
    r"\$\s*\d+|\d+\s*(k|dollars|usd))",
    re.IGNORECASE,
)

RECOVERY_SCAM_PATTERNS = re.compile(
    r"\b("
    r"DM\s+(now|me|us|asap)|"
    r"we\s+are\s+tracing|"
    r"get\s+your\s+money\s+back|"
    r"recover(y|\s+expert|\s+specialist)|"
    r"funds\s+recovery|"
    r"crypto\s+recovery|"
    r"contact\s+us\s+(?:now|asap|today)|"
    r"reach\s+out\s+to\s+(?:us|me)|"
    r"100%\s+(?:guaranteed|recovery|success)|"
    r"lost\s+(?:money|funds|crypto|bitcoin)\s+(?:on|to)\s+\w+\s+(?:scam|scammer)"
    r")\b",
    re.IGNORECASE,
)

BRAND_LIST_PATTERN = re.compile(
    r"(nft\s+scam|bitcoin\s+scam|forex|cfd|binary\s+option|"
    r"wallet\s+drained|bored\s*apes?|ethereum\s+scam)",
    re.IGNORECASE,
)

def is_recovery_scammer(text):
    if not isinstance(text, str):
        return False
    if RECOVERY_SCAM_PATTERNS.search(text):
        return True
    has_contact = bool(re.search(r"\b(DM|dm)\b|contact|reach\s+out", text, re.IGNORECASE))
    brand_hits = len(BRAND_LIST_PATTERN.findall(text))
    return has_contact and brand_hits >= 2

NEWS_HEADLINE_PATTERN = re.compile(
    r"^\W*(leaders?|chief|ceo|president|founder|judge|jury|court|"
    r"police|sentence[d]?|charged|arrested|indicted|"
    r"\d+\s+(?:years?|months?)\s+in\s+prison)\b",
    re.IGNORECASE,
)

PROMO_BLOG_PATTERN = re.compile(
    r"\b("
    r"our\s+(?:recent\s+)?(?:blog|article|post)|"
    r"breaks?\s+down|"
    r"read\s+(?:more|the\s+full)|"
    r"link\s+in\s+bio|"
    r"check\s+(?:out|it)\s+(?:our|this)|"
    r"new\s+(?:blog|article|episode|video)"
    r")\b",
    re.IGNORECASE,
)

def looks_like_news_or_promo(text, n_urls=0):
    if not isinstance(text, str):
        return False
    if PROMO_BLOG_PATTERN.search(text):
        return True
    if n_urls > 0 and NEWS_HEADLINE_PATTERN.search(text):
        return True
    return False

METAPHOR_PATTERNS = re.compile(
    r"\b(life is a scam|love is a scam|system is a scam|"
    r"capitalism is a scam|college is a scam|dating is a scam|"
    r"my life is a scam|everything is a scam|"
    r"life'?s a scam|love'?s a scam)\b",
    re.IGNORECASE,
)

SOFT_POLITICS_PATTERNS = re.compile(
    r"\b(harris|desantis|newsom|kamala|obama|congress|senate|senator|"
    r"republican|democrat|gop|maga|lawmaker|politician|"
    r"voter fraud|election fraud|stolen election|rigged election|"
    r"deep state)\b",
    re.IGNORECASE,
)

BRAND_USERNAMES = {
    "lifelock", "aura", "norton", "nortononline", "mcafee",
    "cnn", "foxnews", "msnbc", "nytimes", "wsj", "reuters",
    "ap", "bbcworld", "abc", "abcnews", "nbcnews", "cbsnews",
}

EMOTIONAL_ONLY = re.compile(
    r"\b(catfish|catfished|catfishing|emotional scam|"
    r"scammed my heart|broke my heart)\b",
    re.IGNORECASE,
)

URL_RE = re.compile(r"https?://\S+")
MENTION_RE = re.compile(r"@\w+")

def clean_for_length(text):
    if not isinstance(text, str):
        return ""
    t = URL_RE.sub("", text)
    t = MENTION_RE.sub("", t)
    return re.sub(r"\s+", " ", t).strip()

us_keywords = [
    "united states", "usa", "u.s.", " us ", " us,", "america",
    "new york", "california", "texas", "florida", "chicago",
    "los angeles", "boston", "seattle", "miami", "atlanta",
    "dallas", "houston", "philadelphia", "phoenix", "denver",
    "washington", "san francisco", "san diego",
]

def looks_us(row):
    if row.get("place_country_code") == "US":
        return True
    loc = " " + str(row.get("user_location") or "").lower() + " "
    return any(k in loc for k in us_keywords)


## AJUSTE 4: Detector de metáfora institucional

Si en una ventana de ≤6 palabras coinciden:
- un **sistema institucional** (social security, stock market, ai, housing market...)
- una **palabra-acusación** (ponzi, scam, fraud, scheme)

→ es metáfora retórica, no descripción de fraude.

La proximidad léxica (≤6 palabras) evita matar tweets como "scammers target Social Security recipients" (donde Social Security va como víctima, no como sujeto).


In [ ]:
INSTITUTIONAL_TARGETS = [
    "social security",
    "stock market",
    "housing market",
    "real estate market",
    r"\bai\b",
    "artificial intelligence",
    "crypto industry",
    "the tax system",
    "tax system",
    "taxes",
    "capitalism",
    "the government",
    "modern slavery",
    "slave engine",
    "marriage",
    "religion",
    "the church",
    "college",
    "the system",
    "the economy",
    "the federal reserve",
    "fiat currency",
    "social welfare",
    "welfare system",
    "healthcare system",
    "education system",
    "school system",
    "pension",
    "401k",
    "wall street",
]

ACCUSATION_WORDS = ["ponzi", "scheme", "scam", "fraud"]

# Compilamos en regex con ventana acotada de 6 palabras (~36 caracteres)
WINDOW = r"(?:\W+\w+){0,6}\W+"

INSTITUTIONAL_METAPHOR_RE = re.compile(
    "|".join(
        f"({inst}{WINDOW}(?:{'|'.join(ACCUSATION_WORDS)}))"
        f"|((?:{'|'.join(ACCUSATION_WORDS)}){WINDOW}{inst})"
        for inst in INSTITUTIONAL_TARGETS
    ),
    re.IGNORECASE,
)

def is_institutional_metaphor(text):
    if not isinstance(text, str):
        return False
    return bool(INSTITUTIONAL_METAPHOR_RE.search(text))


# Test rápido del filtro con ejemplos reales del v3
test_cases = [
    ("Social Security is a Ponzi scheme", True),
    ("Marriage is like Social Security a Ponzi Scheme!", True),
    ("AI is an overvalued ponzi scheme", True),
    ("stock market is a modern-day slave engine... it's a ponzi scheme", True),
    ("housing market is a ponzi scheme", True),
    ("Capitalism is a scam", True),
    # Casos que NO deben filtrarse (fraudes reales que mencionan a un sistema):
    ("Scammers are targeting Social Security recipients in Miami", False),
    ("I lost $5000 in a crypto scam last week", False),
    ("Anyone else getting toll scam text messages?!", False),
    ("Bernie Madoff stole billions in his Ponzi scheme over many years", False),
]
print("=== TEST DEL FILTRO DE METÁFORA INSTITUCIONAL ===\n")
for text, expected in test_cases:
    got = is_institutional_metaphor(text)
    status = "✅" if got == expected else "❌ FALLO"
    print(f"  {status} Esperado={expected:5} Got={got:5}  | {text[:80]}")


## Aplicación de los filtros

In [ ]:
df = df_dedup.copy()

# Auxiliares y filtros v3
df["clean_text"]       = df["text"].apply(clean_for_length)
df["clean_len"]        = df["clean_text"].str.len()
df["n_words"]          = df["clean_text"].str.split().str.len().fillna(0).astype(int)
df["hashtag_ratio"]    = (df["n_hashtags"] / df["n_words"].replace(0, 1)).fillna(0)
df["mention_ratio"]    = (df["n_mentions"] / df["n_words"].replace(0, 1)).fillna(0)
df["is_metaphor"]      = df["text"].fillna("").apply(lambda t: bool(METAPHOR_PATTERNS.search(t)))
df["is_soft_politics"] = df["text"].fillna("").apply(lambda t: bool(SOFT_POLITICS_PATTERNS.search(t)))
df["is_emotional"]     = df["text"].fillna("").apply(lambda t: bool(EMOTIONAL_ONLY.search(t)))
df["is_brand_account"] = df["username"].fillna("").str.lower().isin(BRAND_USERNAMES)
df["likely_us"]        = df.apply(looks_us, axis=1)
df["has_strong_fraud"] = df["text"].fillna("").apply(lambda t: bool(STRONG_FRAUD_TERMS.search(t)))
df["has_money"]        = df["text"].fillna("").apply(lambda t: bool(MONEY_TERMS.search(t)))
df["is_recovery_scammer"] = df["text"].fillna("").apply(is_recovery_scammer)
df["is_news_or_promo"]    = df.apply(lambda r: looks_like_news_or_promo(r["text"], r.get("n_urls", 0)), axis=1)
df["semantically_relevant"] = df["has_strong_fraud"] | df["has_money"]

# NUEVO en v4
df["is_institutional_metaphor"] = df["text"].fillna("").apply(is_institutional_metaphor)

print("=== DISTRIBUCIÓN DE FLAGS (v4) ===")
print(f"Total tweets:                       {len(df)}")
print(f"Semánticamente relevantes:          {df['semantically_relevant'].sum()}")
print(f"Recovery scammers:                  {df['is_recovery_scammer'].sum()}")
print(f"Noticias / blog promos:             {df['is_news_or_promo'].sum()}")
print(f"Metáforas institucionales (NUEVO):  {df['is_institutional_metaphor'].sum()} ({df['is_institutional_metaphor'].mean()*100:.1f}%)")
print(f"Probablemente US:                   {df['likely_us'].sum()}")


In [ ]:
# Aplicar todos los filtros
mask = (
    (df["clean_len"] >= 40) &
    (df["semantically_relevant"]) &
    (df["likely_us"]) &
    (df["hashtag_ratio"] < 0.3) &
    (df["mention_ratio"] < 0.4) &
    (~df["is_metaphor"]) &
    (~df["is_soft_politics"]) &
    (~df["is_emotional"]) &
    (~df["is_brand_account"]) &
    (~df["is_recovery_scammer"]) &
    (~df["is_news_or_promo"]) &
    (~df["is_institutional_metaphor"])    # nuevo
)

df_clean_v4 = df[mask].reset_index(drop=True)

print("=== RESUMEN DE LIMPIEZA v4 ===")
print(f"Brutos deduplicados:      {len(df)}")
print(f"Limpios finales (v4):     {len(df_clean_v4)}")
print(f"Tasa de retención:        {len(df_clean_v4) / max(len(df), 1) * 100:.1f}%")
print()
print("Por categoría temática:")
for label in ["phishing", "payment_apps", "crypto", "romance_monetary", "impersonation"]:
    n = df_clean_v4["query_labels"].fillna("").str.contains(label).sum()
    print(f"  {label:<20} {n:>6}")


## Inspección: comparativa v3 vs v4

Cuántos tweets quita el ajuste 4 y cuáles son.


In [ ]:
# Tweets que pasaban v3 pero NO pasan v4 (los que ahora descartamos)
mask_v3 = (
    (df["clean_len"] >= 40) &
    (df["semantically_relevant"]) &
    (df["likely_us"]) &
    (df["hashtag_ratio"] < 0.3) &
    (df["mention_ratio"] < 0.4) &
    (~df["is_metaphor"]) &
    (~df["is_soft_politics"]) &
    (~df["is_emotional"]) &
    (~df["is_brand_account"]) &
    (~df["is_recovery_scammer"]) &
    (~df["is_news_or_promo"])
)
df_v3 = df[mask_v3]

newly_excluded = df_v3[~df_v3["tweet_id"].isin(df_clean_v4["tweet_id"])]
print(f"=== TWEETS DESCARTADOS NUEVOS por el ajuste 4: {len(newly_excluded)} ===\n")
for _, row in newly_excluded.sample(min(15, len(newly_excluded)), random_state=42).iterrows():
    print(f"[{row['query_labels']}] @{row['username']}")
    print(f"  {row['text'][:280]}")
    print()


In [ ]:
# Muestra final v4
print("=== MUESTRA ALEATORIA DE 20 TWEETS LIMPIOS v4 ===\n")
for _, row in df_clean_v4.sample(min(20, len(df_clean_v4)), random_state=42).iterrows():
    print(f"[{row['query_labels']}] @{row['username']} — {row['user_location']}")
    print(f"  {row['text'][:280]}")
    print()


## Guardado

In [ ]:
import os
os.makedirs("../data/raw", exist_ok=True)
df_clean_v4.to_csv("../data/raw/scam_us_clean_v4.csv", index=False, encoding="utf-8")
print(f"Guardado: ../data/raw/scam_us_clean_v4.csv ({len(df_clean_v4)} filas)")
